# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIRˆ² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. You will learn how to:
 - Load FAIR data metadata and records for analysis
 - Enumerate record sets and fields by their `@id`
 - Extract records to pandas DataFrames
 - Perform exploratory data analysis (EDA), filtering, normalization and grouping
 - Visualize attribute distributions

### Dataset Source

The dataset is defined by a [Croissant schema](https://mlcommons.org/croissant/), accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and acquire a high-level summary using `mlcroissant`. This also demonstrates proper handling of the Dataset object (do not subscript or iterate over it).

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Date Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview

Enumerate available record sets, fields, and their `@id`s. We use the `record_sets` attribute of the Croissant metadata to discover available structured data. 

The demonstrated technique ensures reproducibility and clarity by always referencing entities by their Croissant `@id`.

In [ ]:
# List all available record sets in the dataset by their `@id` and name
record_sets = getattr(metadata, 'record_sets', [])
if not record_sets:
    record_sets = getattr(metadata, 'recordSet', [])  # fallback for alternate field names

print(f"Number of record sets: {len(record_sets)}\n")

record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"Record Set: @id = {rs_id}, name = {rs_name}")
    record_set_ids.append(rs_id)

# For demonstration, print fields for each record set
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    fields = getattr(rs, 'fields', [])
    if not fields:
        fields = getattr(rs, 'field', [])  # fallback for alternate field names
    print(f"\nFields for record set '{rs_name}' (@id: {rs_id}):")
    for fld in fields:
        fld_id = getattr(fld, '@id', None)
        fld_name = getattr(fld, 'name', None)
        print(f"  Field: @id = {fld_id}, name = {fld_name}")

## 3. Data Extraction

Extract all records from each record set into a pandas DataFrame. Use the `@id` of each record set for reference. The record sets and fields discovered above are used here.

> **Note**: When using the Croissant API, always reference record sets and fields by their `@id` exactly as shown in the overview.

In [ ]:
# Collect all data into DataFrames, keyed by record set @id
dataframes = {}

# If there are no record sets, the dataset is likely flat/tabular: try default record set id
if len(record_set_ids) == 0:
    # Many tabular Croissant datasets define a single record set with '@id' of the schema url + '#default-record-set' or similar
    inferred_rs_id = croissant_url + '#default-record-set'
    record_set_ids = [inferred_rs_id]

for record_set_id in record_set_ids:
    try:
        # Iterate and collect all records for the record set
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set '@id': {record_set_id}")
        if not dataframes[record_set_id].empty:
            print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())
    except Exception as e:
        print(f"Could not load records for record set '@id': {record_set_id} ({e})")

## 4. Exploratory Data Analysis (EDA)

Apply exploratory steps such as filtering or normalizing numeric fields. All column accesses *must use their field `@id`*. The demonstration below assumes at least one numeric field is present; adapt the field `@id` as appropriate from your data summary above.

In [ ]:
# Example setup
# (replace with your actual record set @id and numeric field @id from step 2 above)
record_set_id = record_set_ids[0] if record_set_ids else None
# List columns and try to choose a numeric column (e.g., Age variable or an interval)
if record_set_id:
    df = dataframes[record_set_id]
    print("DataFrame columns:", df.columns.tolist())

    # Try automatically inferring a likely numeric column (fallback to manual)
    import numpy as np
    numeric_candidates = []
    for col in df.columns:
        try:
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notnull().sum() > 0:
                numeric_candidates.append(col)
        except Exception:
            continue
    print("Detected numeric field @id candidates:", numeric_candidates)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # choose the first detected numeric field

        # Filter records
        threshold = df[numeric_field_id].astype(float).median()  # Use median as meaningful threshold
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping: try to find a suitable group/categorical field
        # Candidates: non-numeric, e.g. sex, subtype, diagnosis
        group_candidates = [col for col in df.columns if col != numeric_field_id and not np.issubdtype(df[col].dtype, np.number)]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")

## 5. Visualization

Visualize the distribution of a selected numeric field (referenced by its canonical Croissant `@id`), optionally separated by a key attribute.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    if group_candidates:
        group_field_id = group_candidates[0]
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook showed how to load and explore the FAIRˆ² dataset using `mlcroissant`. By following Croissant's robust `@id` scheme, we:
- Inspected and referenced all dataset entities by ID
- Extracted and displayed record sets
- Performed filtering and normalization on numeric fields
- Visualized field distributions and relationships

For project- or analysis-specific deep-dives, adapt the record set and field `@id`s in the EDA and visualization sections to your needs for precise, reproducible FAIR data science.